In [ ]:
from graphein.ml.conversion import GraphFormatConvertor
import numpy as np
import torch
from sklearn.model_selection import train_test_split
from graphein.protein.utils import get_obsolete_mapping
import pandas as pd
import os
from tqdm.notebook import tqdm

In [ ]:
from tqdm.notebook import tqdm
import time

for i in tqdm(range(10)):
    time.sleep(0.1)

In [ ]:
# CONFIGS
import graphein.protein as gp
from functools import partial
from graphein.ml.conversion import GraphFormatConvertor
from graphein.protein.edges.distance import (add_peptide_bonds,
                                             add_hydrogen_bond_interactions,
                                             add_disulfide_interactions,
                                             add_ionic_interactions,
                                             add_aromatic_interactions,
                                             add_aromatic_sulphur_interactions,
                                             add_cation_pi_interactions
                                             )


# 1: Distance-based
dist_edge_func = {"edge_construction_functions": [partial(gp.add_distance_threshold, threshold=5, long_interaction_threshold=0)]}

# 2: Biochemical interactions, select set
select_edge_func = {"edge_construction_functions": [add_peptide_bonds,
                                                    add_hydrogen_bond_interactions,
                                                    add_disulfide_interactions,
                                                    add_ionic_interactions,
                                                    gp.add_salt_bridges]}

# 3: Biochemical interactions, expanded set
all_edge_func = {"edge_construction_functions": [add_peptide_bonds,
                                                 add_aromatic_interactions,
                                                 add_hydrogen_bond_interactions,
                                                 add_disulfide_interactions,
                                                 add_ionic_interactions,
                                                 add_aromatic_sulphur_interactions,
                                                 add_cation_pi_interactions,
                                                 gp.add_hydrophobic_interactions,
                                                 gp.add_vdw_interactions,
                                                 gp.add_backbone_carbonyl_carbonyl_interactions,
                                                 gp.add_salt_bridges]}

In [ ]:

# A: Just one-hot encodings
one_hot = {"node_metadata_functions" : [gp.amino_acid_one_hot]}

# B: Selected biochemical features
all_graph_metadata = {"graph_metadata_functions" : [gp.rsa,
                                                    gp.secondary_structure]}
all_node_metadata = {"node_metadata_functions" : [gp.amino_acid_one_hot,
                                                  gp.meiler_embedding,
                                                  partial(gp.expasy_protein_scale, add_separate=True)],
                     "dssp_config": gp.DSSPConfig()}


config_1A = gp.ProteinGraphConfig(**{**dist_edge_func, **one_hot})
config_1B = gp.ProteinGraphConfig(**{**dist_edge_func, **all_graph_metadata, **all_node_metadata})

config_2A = gp.ProteinGraphConfig(**{**select_edge_func, **one_hot})
config_2B = gp.ProteinGraphConfig(**{**select_edge_func, **all_graph_metadata, **all_node_metadata})

config_3A = gp.ProteinGraphConfig(**{**all_edge_func, **one_hot})
config_3B = gp.ProteinGraphConfig(**{**all_edge_func, **all_graph_metadata, **all_node_metadata})

In [ ]:
# Plotting
from graphein.protein.graphs import construct_graph
from graphein.protein.visualisation import plotly_protein_structure_graph

g1 = construct_graph(config=config_1A, path="6RAI.pdb")
g2 = construct_graph(config=config_2A, path="6RAK.pdb")
g3 = construct_graph(config=config_3A, path="6RVC.pdb")

p1 = plotly_protein_structure_graph(
    g1,
    colour_edges_by="kind",
    colour_nodes_by="degree",
    label_node_ids=False,
    plot_title="",
    node_size_multiplier=1
)
p2 = plotly_protein_structure_graph(
    g2,
    colour_edges_by="kind",
    colour_nodes_by="degree",
    label_node_ids=False,
    plot_title="",
    node_size_multiplier=1
)
p3 = plotly_protein_structure_graph(
    g3,
    colour_edges_by="kind",
    colour_nodes_by="degree",
    label_node_ids=False,
    plot_title="",
    node_size_multiplier=1
)

In [ ]:
p1.show(); p2.show(); p3.show()